Phase 0: 初始化 Jupyter Notebook

In [12]:
# ==========================================
# 🛰️ ARIA v5.0 - Matai'an Three-Act Auditor
# ==========================================
import os
from dotenv import load_dotenv
import pystac_client
import planetary_computer as pc
import stackstac
import rioxarray as rxr
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path

# 1. 載入環境變數
load_dotenv()

STAC_ENDPOINT = os.getenv("STAC_ENDPOINT")
BBOX_STR = os.getenv("MATAIAN_BBOX")
MATAIAN_BBOX = [float(x) for x in BBOX_STR.split(',')] if BBOX_STR else [121.28, 23.56, 121.52, 23.76]

# 2. 確保輸出目錄存在
Path("output").mkdir(parents=True, exist_ok=True)

# 3.載入環境變數 (dotenv 寫法)
import os
from dotenv import load_dotenv

# 1. 讀取 .env 檔案
load_dotenv()

# 2. 讀取一般字串變數
STAC_ENDPOINT = os.getenv("STAC_ENDPOINT")
S2_COLLECTION = os.getenv("S2_COLLECTION")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
CWA_API_KEY = os.getenv("CWA_API_KEY")

# 3. 讀取時間區間
PRE_EVENT_START = os.getenv("PRE_EVENT_START")
PRE_EVENT_END = os.getenv("PRE_EVENT_END")
MID_EVENT_START = os.getenv("MID_EVENT_START")
MID_EVENT_END = os.getenv("MID_EVENT_END")
POST_EVENT_START = os.getenv("POST_EVENT_START")
POST_EVENT_END = os.getenv("POST_EVENT_END")

# 4. 讀取並轉換數值變數 (加上預設值防呆)
TARGET_EPSG = int(os.getenv("TARGET_EPSG", 32651))
S2_CLOUD_MAX = int(os.getenv("S2_CLOUD_MAX", 20))

# 5. 讀取並轉換列表 (List) 變數
# 將 "121.28,23.56,121.52,23.76" 轉換為 [121.28, 23.56, 121.52, 23.76]
bbox_str = os.getenv("MATAIAN_BBOX")
MATAIAN_BBOX = [float(x) for x in bbox_str.split(',')] if bbox_str else [121.28, 23.56, 121.52, 23.76]

# 將 "B02,B03,B04,B08,B11,B12" 轉換為 ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']
bands_str = os.getenv("S2_BANDS")
S2_BANDS = bands_str.split(',') if bands_str else ["B02", "B03", "B04", "B08", "B11", "B12"]


# 測試印出，確認讀取正確 (不要印出 API Key 以保安全)
print("✅ 環境變數載入成功！")
print(f"📦 BBox: {MATAIAN_BBOX} (型態: {type(MATAIAN_BBOX[0])})")
print(f"🛰️ 波段: {S2_BANDS} (型態: {type(S2_BANDS)})")
print(f"🎯 EPSG: {TARGET_EPSG} (型態: {type(TARGET_EPSG)})")

✅ 環境變數載入成功！
📦 BBox: [121.28, 23.56, 121.52, 23.76] (型態: <class 'float'>)
🛰️ 波段: ['B02', 'B03', 'B04', 'B08', 'B11', 'B12'] (型態: <class 'list'>)
🎯 EPSG: 32651 (型態: <class 'int'>)


Phase 1: 環境設定與 STAC 影像篩選 (對應作業 3A)
首先，透過 Planetary Computer 獲取三個時期的無雲影像 ID。

In [17]:
# 1. 初始化 STAC 客戶端
catalog = pystac_client.Client.open(STAC_ENDPOINT, modifier=pc.sign_inplace)

# 2. 定義搜尋函數
def get_best_item_id(start_date, end_date, cloud_limit):
    search = catalog.search(
        collections=[S2_COLLECTION],
        bbox=MATAIAN_BBOX,
        datetime=f"{start_date}/{end_date}",
        query={"eo:cloud_cover": {"lt": cloud_limit}}
    )
    items = search.item_collection()
    if not items:
        raise ValueError(f"在 {start_date} 至 {end_date} 期間找不到符合雲量限制的影像。")
    best_item = min(items, key=lambda i: i.properties["eo:cloud_cover"])
    return best_item.id

# 3. 獲取 ID
PRE_ITEM_ID = get_best_item_id(PRE_EVENT_START, PRE_EVENT_END, S2_CLOUD_MAX)
MID_ITEM_ID = get_best_item_id(MID_EVENT_START, MID_EVENT_END, 40)  # 季風期手動放寬至 40%
POST_ITEM_ID = get_best_item_id(POST_EVENT_START, POST_EVENT_END, 30)

print(f"✅ 已鎖定影像：\nPre: {PRE_ITEM_ID}\nMid: {MID_ITEM_ID}\nPost: {POST_ITEM_ID}")

✅ 已鎖定影像：
Pre: S2A_MSIL2A_20250615T023141_R046_T51QUG_20250615T070417
Mid: S2C_MSIL2A_20250911T022551_R046_T51QUG_20250911T055914
Post: S2B_MSIL2A_20251016T022559_R046_T51QUG_20251016T042804


Phase 2: 載入資料與計算光譜變量 (對應作業 3B)
將三個時期的影像串流為 xarray DataArray，並定義四大變量計算函數。

In [19]:
# 1. 建立影像方塊 (Cubes)
s2_coll = catalog.get_collection(S2_COLLECTION)

def stream_cube(item_id):
    item = s2_coll.get_item(item_id)
    return stackstac.stack(
        [item], assets=S2_BANDS, epsg=TARGET_EPSG, resolution=10, bounds_latlon=MATAIAN_BBOX
    ).squeeze()

pre_cube = stream_cube(PRE_ITEM_ID)
mid_cube = stream_cube(MID_ITEM_ID)
post_cube = stream_cube(POST_ITEM_ID)

# 2. 定義指標
def calc_ndvi(c): return (c.sel(band="B08") - c.sel(band="B04")) / (c.sel(band="B08") + c.sel(band="B04"))
def calc_bsi(c):
    b11, b04, b08, b02 = c.sel(band="B11"), c.sel(band="B04"), c.sel(band="B08"), c.sel(band="B02")
    return ((b11 + b04) - (b08 + b02)) / ((b11 + b04) + (b08 + b02))

# 3. 計算變化量
nir_drop = pre_cube.sel(band="B08") - post_cube.sel(band="B08")
swir_post = post_cube.sel(band="B12")
bsi_change = calc_bsi(post_cube) - calc_bsi(pre_cube)
ndvi_change = calc_ndvi(pre_cube) - calc_ndvi(post_cube)

Phase 3: 閾值調校與遮罩生成 (對應作業 3C)
依據物理特性設定條件式。注意混濁水體的特性。

In [20]:
import numpy as np
from rasterio.features import shapes
import shapely.geometry

def vectorize_mask(mask_da, min_area=2000):
    mask_np = mask_da.values.astype(np.uint8)
    results = ({'properties': {'val': v}, 'geometry': s} for i, (s, v) in enumerate(shapes(mask_np, mask=mask_np==1, transform=mask_da.rio.transform())))
    geoms = [shapely.geometry.shape(r['geometry']) for r in results]
    gdf = gpd.GeoDataFrame(geometry=geoms, crs=f"EPSG:{TARGET_EPSG}")
    return gdf[gdf.area > min_area]

# A. 堰塞湖 (注意：混濁水體 NIR 門檻設為 0.18)
lake_gdf = vectorize_mask((pre_cube.sel(band="B08") > 0.25) & (mid_cube.sel(band="B08") < 0.18))

# B. 崩塌源 (NIR 下降且 SWIR 升高)
ls_gdf = vectorize_mask((nir_drop > 0.15) & (swir_post > 0.25))

# C. 土石流沉積 (NDVI 下降且 BSI 升高)
debris_gdf = vectorize_mask((ndvi_change > 0.25) & (bsi_change > 0.10))

Phase 4: 空間檢核與衝擊分析表 (對應作業 3D)
載入先前的向量圖層，進行空間交集 (Spatial Join)。

In [23]:
import pandas as pd

# 假設你已將遮罩轉為 GeoDataFrame: gdf_lake, gdf_landslide, gdf_debris (皆為 EPSG:3826)
# 載入設施資料
shelters = gpd.read_file("data/shelters_hualien.gpkg")
guangfu = gpd.read_file("data/guangfu_overlay.gpkg")
bottlenecks = gpd.read_file("data/xiulin_network.gpkg")

# 合併所有設施進行檢核
all_assets = pd.concat([shelters, guangfu, bottlenecks], ignore_index=True)

# 空間檢核邏輯 (以 Debris Flow 為例)
# 使用 sjoin 檢查設施是否落在土石流多邊形內
hit_debris = gpd.sjoin(all_assets, gdf_debris, how="inner", predicate="intersects")
all_assets['Debris Flow Hit (Y/N)'] = all_assets['name'].isin(hit_debris['name']).map({True: 'Y', False: 'N'})

# 依此類推完成 Landslide 與 Lake 的檢核，並整理為 DataFrame 輸出 impact_table.csv

c:\anaconda3\envs\gis-env\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'guangfu_overlay.gpkg': 'guangfu' (default), 'shelters', 'bottlenecks', 'guangfu_nodes'. Specify layer parameter to avoid this warning.
  result = read_func(
c:\anaconda3\envs\gis-env\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'xiulin_network.gpkg': 'nodes' (default), 'edges'. Specify layer parameter to avoid this warning.
  result = read_func(


ValueError: Cannot determine common CRS for concatenation inputs, got ['TWD97 / TM2 zone 121', 'WGS 84']. Use `to_crs()` to transform geometries to the same CRS before merging.